---
title: "Omeka S API: Uploading in Bulk"
---

The examples in this section elaborate patterns that are illustrated in the section on uploading an individual item. Code in this section was significantly enhanced with input from an automated coding assistant.[^codernote]

[^codernote]: Initial enhancements were made in 2024, with the assistance of Claude Opus 4.X.

In [ ]:
import requests
import csv
import json
import os

In [ ]:
def import_omeka_items(csv_path, media_directory, site_api, item_set_id, key_identity, key_credential, column_to_term):
    """
    Import items and their associated media files to an Omeka S site via API.
    
    Parameters:
    - csv_path: Path to the CSV file containing item metadata
    - media_directory: Directory containing media files
    - site_api: Base URL of the Omeka S API
    - item_set_id: ID of the Omeka S item set to add items to
    - key_identity: API key identity
    - key_credential: API key credential
    - column_to_term: dictionary mapping CSV column names to vocabulary terms
      (e.g. {'title': 'dcterms:title'}); columns not listed are skipped
    
    Returns:
    - List of response statuses for each item import
    """
    # API endpoint for items
    action = '/items'
    
    # Prepare API authentication parameters
    params = {
        'key_credential': key_credential,
        'key_identity': key_identity,
    }
    
    # Store import results
    import_results = []
    
    # Read the CSV file
    with open(csv_path, 'r', encoding='utf-8') as csvfile:
        # Assume the first row contains headers
        reader = csv.DictReader(csvfile)
        
        # Process each row in the CSV
        for row_index, row in enumerate(reader):
            # Prepare files dictionary for this item
            files = {}
            
            # Construct item metadata
            item_metadata = {
                "@type": "o:Item",
                "o:item_set": [{"o:id": item_set_id}],
                "o:media": []
            }
            
            # Add the metadata fields listed in column_to_term.
            # "property_id": "auto" tells Omeka S to look up the property
            # from the term used as the key (e.g. dcterms:date), so each
            # column is saved to its own field. Columns not in the mapping
            # are skipped, since Omeka S has no property for an unknown term.
            for column, term in column_to_term.items():
                value = row.get(column)
                if not value:
                    continue
                item_metadata[term] = [{
                    "property_id": "auto",
                    "@value": value,
                    "type": "literal"
                }]
            
            # Find and add media files
            # Assumes there's a unique identifier in the CSV that matches part of the media filename
            media_files = []
            for filename in os.listdir(media_directory):
                # Customize the matching logic based on your file naming convention
                if any(identifier.lower() in filename.lower() for identifier in row.values()):
                    media_path = os.path.join(media_directory, filename)
                    media_files.append(media_path)
            
            # Prepare media files for upload
            for file_index, media_file_path in enumerate(media_files):
                files[f'file[{file_index}]'] = open(media_file_path, 'rb')
                
                # Add media metadata
                item_metadata['o:media'].append({
                    "o:ingester": "upload",
                    "file_index": str(file_index),
                    "o:item": {},
                    "dcterms:title": [{
                        "property_id": "auto",
                        "@value": f"Media file for {row.get('title', 'item')}",
                        "type": "literal"
                    }]
                })
            
            # Add item metadata to files
            files['data'] = (None, json.dumps(item_metadata))
            
            try:
                # Send API request to create item
                r = requests.post(site_api + action, params=params, files=files)
                import_results.append({
                    'row': row_index,
                    'status_code': r.status_code,
                    'response': r.text
                })
                
                # Close any open file handles
                for file_obj in files.values():
                    if hasattr(file_obj, 'close'):
                        file_obj[1].close()
            
            except Exception as e:
                import_results.append({
                    'row': row_index,
                    'error': str(e)
                })
    
    return import_results

In [ ]:
# Example usage
# map CSV columns to vocabulary terms; base this on your MAP
example_column_to_term = {
    'title': 'dcterms:title',
    'date': 'dcterms:date',
}

if __name__ == '__main__':
    results = import_omeka_items(
        csv_path='path/to/your/metadata.csv',
        media_directory='path/to/media/files',
        site_api='http://your-omeka-s-site/api',
        item_set_id=100, # input your set ID here, if desired
        key_identity='key-identity-for-your-API-user-agent',
        key_credential='key-credential-for-your-API-user-agent',
        column_to_term=example_column_to_term
    )
    
    # Print import results
    for result in results:
        print(result)

In [ ]:
# usage in practice
with open('../omeka-credentials.json', 'r') as credentials:
    keys = json.load(credentials)
key_identity, key_credential = keys['key_identity'], keys['key_credential']

# map the columns of the transformed CSV to Dublin Core terms,
# following the draft crosswalk in the MAP chapter
column_to_term = {
    'title': 'dcterms:title',
    'date': 'dcterms:date',
    'item_id': 'dcterms:identifier',
    'dig_format': 'dcterms:format',
    'rights': 'dcterms:rights',
    'source_url': 'dcterms:source',
}

if __name__ == '__main__':
    results = import_omeka_items(
        csv_path='../collection-project/collection_items_data.csv',
        media_directory='../collection-project/item-files',
        site_api='https://your-omeka-s-site.example/omeka-s/api',
        item_set_id=654,
        key_identity=key_identity,
        key_credential=key_credential,
        column_to_term=column_to_term
    )
    
    # Print import results
    for result in results:
        print(result)